In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.DataFrame({
    'Age_Group': [1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5],
    'Ig_A': [83, 85, 84, 85, 85, 86, 86, 87, 86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90, 89, 90, 90, 91, 90, 92]
})

data['Age_Group'] = data['Age_Group'].astype('category')

print(data.head())

  Age_Group  Ig_A
0         1    83
1         1    85
2         2    84
3         2    85
4         2    85


In [3]:
model = ols('Ig_A ~ C(Age_Group)', data=data).fit()

print("Регрессионная модель")
print(f"R-squared (Коэффициент детерминации): {model.rsquared:.4f}")
print(f"P-value для всей модели: {model.f_pvalue:.2e}\n")

print("Коэффициенты регрессии")
coef_df = pd.DataFrame({
    'Коэффициент': model.params,
    'Std Err': model.bse,
    'P-value': model.pvalues
})
print(coef_df.to_string(float_format="%.4f"))

if model.f_pvalue < 0.05:
    print("P-value модели строго меньше 0.05. Возраст оказывает статистически значимое влияние на содержание иммуноглобулина в крови.")
else:
    print("Модель не значима.")

Регрессионная модель
R-squared (Коэффициент детерминации): 0.8106
P-value для всей модели: 5.41e-07

Коэффициенты регрессии
                   Коэффициент  Std Err  P-value
Intercept              84.0000   0.7605   0.0000
C(Age_Group)[T.2]       1.5000   0.8782   0.1031
C(Age_Group)[T.3]       3.8182   0.8268   0.0002
C(Age_Group)[T.4]       6.0000   0.9315   0.0000
C(Age_Group)[T.5]       7.0000   1.0756   0.0000
P-value модели строго меньше 0.05. Возраст оказывает статистически значимое влияние на содержание иммуноглобулина в крови.


In [4]:
anova_table = sm.stats.anova_lm(model, typ=2)
print("Таблица ANOVA:")
print(anova_table)

Таблица ANOVA:
                 sum_sq    df     F        PR(>F)
C(Age_Group)  99.023636   4.0  21.4  5.407435e-07
Residual      23.136364  20.0   NaN           NaN


In [5]:
# Выполняем попарное сравнение с поправкой Холма-Бонферрони
comparisons = model.t_test_pairwise('C(Age_Group)', method='holm')
results_df = comparisons.result_frame.copy()

keep_cols = ['coef', 'std err', 't', 'P>|t|', 'pvalue-holm', 'reject-holm']
results_df = results_df[keep_cols]

results_df.rename(columns={
    'pvalue-holm': 'P-value (Holm)', 
    'reject-holm': 'Significant (p<0.05)'
}, inplace=True)

print("Результаты попарного сравнения (с поправкой Холма-Бонферрони):")
print("-" * 80)
print(results_df.sort_values(by="P-value (Holm)").to_string(float_format="%.5f"))

Результаты попарного сравнения (с поправкой Холма-Бонферрони):
--------------------------------------------------------------------------------
       coef  std err       t   P>|t|  P-value (Holm)  Significant (p<0.05)
4-1 6.00000  0.93146 6.44152 0.00000         0.00002                  True
5-1 7.00000  1.07555 6.50827 0.00000         0.00002                  True
4-2 4.50000  0.69427 6.48165 0.00000         0.00002                  True
5-2 5.50000  0.87819 6.26290 0.00000         0.00003                  True
3-1 3.81818  0.82679 4.61810 0.00017         0.00100                  True
3-2 2.31818  0.54586 4.24681 0.00040         0.00198                  True
5-3 3.18182  0.82679 3.84842 0.00100         0.00401                  True
4-3 2.18182  0.62799 3.47430 0.00239         0.00718                  True
2-1 1.50000  0.87819 1.70806 0.10310         0.20620                 False
5-4 1.00000  0.93146 1.07359 0.29579         0.29579                 False
